# 06 · 효율 측정 — Mamba(선형) vs Transformer(이차)

Intro/Abstract 에서 **"Mamba 는 선형 시간·고정 크기 상태라 긴 시퀀스에서 효율적"** 을
Mamba 채택의 근거로 썼다 -> **수치가 없으면 그 문장이 공허해진다.** 여기서 채운다.

- 측정: 청크 1회 생성의 **latency** 와 **peak VRAM** 을 **K(청크 길이)** 를 늘려가며.
- 비교: `act`(Transformer dec) vs `ours`(Mamba dec).
- 이미지 없이 **env-state 입력**만 -> ResNet 백본 비용을 빼고 **디코더 비용만** 분리.
- 학습 불필요(랜덤 초기화 정책의 forward) -> **싸고 임팩트 큼**.

주의: CUDA + mamba_ssm 필요 (클러스터에서 실행).


In [ ]:
import sys, time, json
from pathlib import Path
_here = Path.cwd()
_root = next(c for c in (_here, *_here.parents)
             if (c / 'common_final.py').exists() or (c / 'notebooks' / 'common_final.py').exists())
_root = _root / 'notebooks' if (_root / 'notebooks' / 'common_final.py').exists() else _root
sys.path.insert(0, str(_root))
import torch
import common_final as cf

from lerobot.configs.types import FeatureType, PolicyFeature
from lerobot.policies.factory import get_policy_class, make_policy_config
from lerobot.utils.constants import ACTION, OBS_ENV_STATE, OBS_STATE

assert torch.cuda.is_available(), 'CUDA 필요 (mamba_ssm 커널)'
DEV = 'cuda'
DS, DE, DA = 14, 6, 14           # state / env-state / action dim (aloha 기준)
KS = [50, 100, 200, 400, 800]    # 청크 길이 sweep
REPEAT, WARMUP = 20, 5
print('device:', torch.cuda.get_device_name(0), '| K sweep:', KS)

## 정책 빌드 + 측정 루프

In [ ]:
def _cast(v):
    if v in ('true', 'false'):
        return v == 'true'
    try:
        return int(v)
    except ValueError:
        pass
    try:
        return float(v)
    except ValueError:
        return v

def build(tag, K):
    policy_type, _lr, _K, extra, _cp = cf.v23.MODEL_CONFIGS[tag]
    kw = dict(
        input_features={
            OBS_STATE: PolicyFeature(type=FeatureType.STATE, shape=(DS,)),
            OBS_ENV_STATE: PolicyFeature(type=FeatureType.ENV, shape=(DE,)),
        },
        output_features={ACTION: PolicyFeature(type=FeatureType.ACTION, shape=(DA,))},
        chunk_size=K, n_action_steps=K, dropout=0.0,
    )
    for e in extra:                      # 태그의 정책 플래그(carry/bimamba/overlap...) 반영
        k, v = e.replace('--policy.', '').split('=')
        kw[k] = _cast(v)
    cfg = make_policy_config(policy_type, **kw)
    return get_policy_class(policy_type)(cfg).to(DEV).eval()

@torch.no_grad()
def measure(tag, K):
    policy = build(tag, K)
    batch = {OBS_STATE: torch.randn(1, DS, device=DEV),
             OBS_ENV_STATE: torch.randn(1, DE, device=DEV)}
    for _ in range(WARMUP):
        policy.predict_action_chunk(batch)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    t0 = time.perf_counter()
    for _ in range(REPEAT):
        policy.predict_action_chunk(batch)
    torch.cuda.synchronize()
    ms = (time.perf_counter() - t0) / REPEAT * 1000
    mem = torch.cuda.max_memory_allocated() / 2**20
    del policy
    torch.cuda.empty_cache()
    return ms, mem

TAGS = ['act', 'ours']
res = {t: {'K': [], 'ms': [], 'mem': []} for t in TAGS}
for t in TAGS:
    for K in KS:
        ms, mem = measure(t, K)
        res[t]['K'].append(K)
        res[t]['ms'].append(ms)
        res[t]['mem'].append(mem)
        print(f'{t:<6} K={K:<5} {ms:7.2f} ms  {mem:8.1f} MiB')

## 표 + 그림 -> `outputs/final/efficiency/`

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
                     'font.size': 13, 'axes.grid': True, 'grid.alpha': 0.3})

out = cf.OUTPUT_BASE / 'efficiency'
out.mkdir(parents=True, exist_ok=True)
json.dump(res, open(out / 'efficiency.json', 'w'), indent=2)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for t in TAGS:
    c = cf.COLOR.get(t, '#333')
    lab = cf.FINAL_LABELS.get(t, t)
    axes[0].plot(res[t]['K'], res[t]['ms'], '-o', color=c, label=lab)
    axes[1].plot(res[t]['K'], res[t]['mem'], '-o', color=c, label=lab)
axes[0].set(xlabel='chunk size K', ylabel='latency / chunk (ms)', title='inference time (lower=better)')
axes[1].set(xlabel='chunk size K', ylabel='peak VRAM (MiB)', title='memory (lower=better)')
for a in axes:
    a.legend()
fig.suptitle('Decoder efficiency vs chunk length: attention O(L^2) vs Mamba O(L)', fontweight='bold')
fig.savefig(out / 'efficiency.png')
fig.savefig(out / 'efficiency.pdf')
plt.show()

print(f"{'K':>6}" + ''.join(f'{t:>14}' for t in TAGS))
for i, K in enumerate(KS):
    print(f'{K:>6}' + ''.join(f"{res[t]['ms'][i]:>10.1f} ms" for t in TAGS))
print('\n저장:', out)

### 논문에 쓰는 법
- **Fig.** latency/VRAM vs K -> ACT 는 K 커질수록 급증(이차), ours 는 완만(선형).
- **문장**: "K=800 에서 ACT 대비 latency [XX]x, VRAM [XX]x 절감" -> Intro 의 효율 주장 뒷받침.
- 주의: **디코더 비용만** 분리한 수치(이미지 백본 제외) -- 캡션에 명시할 것.
